In [1]:
import os, sys

sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import seaborn as sns
from pathlib import Path
import pandas as pd
from collections import defaultdict

sns.set_theme(style="whitegrid")

In [3]:
with h5py.File(r"E:\Dai hoc\2526I\dacn\flow-matching\data\holdout_hcd.hdf5") as f:
    print(f.keys())
    seqs = f["sequence_integer"][:]
    intensities = f["intensities_raw"][:]
    intensities = np.array(intensities, dtype=np.float32)
    charge_oh = f["precursor_charge_onehot"][:]
charges = np.argmax(charge_oh, axis=1) + 1

<KeysViewHDF5 ['collision_energy', 'collision_energy_aligned', 'collision_energy_aligned_normed', 'intensities_raw', 'masses_pred', 'masses_raw', 'method', 'precursor_charge_onehot', 'rawfile', 'reverse', 'scan_number', 'score', 'sequence_integer']>


In [7]:
charges.shape

(754215,)

In [6]:


# Trim sequences by non-zero length
seq_lens = np.count_nonzero(seqs, axis=1)
N, M = intensities.shape

# Map key -> list where key is (seq_str, charge)
groups = defaultdict(list)

for i in range(N):
    ln = int(seq_lens[i])
    if ln == 0:
        continue
    seq_trim = tuple(int(x) for x in seqs[i, :ln])
    ch = int(charges[i])
    key = (seq_trim, ch)
    if key not in groups.keys():
        groups[key] = []
    groups[key].append(intensities[i])
    if (i + 1) % 50000 == 0:
        print(f'Processed {i+1} / {N}')

Processed 50000 / 754215
Processed 100000 / 754215
Processed 150000 / 754215
Processed 200000 / 754215
Processed 250000 / 754215
Processed 300000 / 754215
Processed 350000 / 754215
Processed 400000 / 754215
Processed 450000 / 754215
Processed 500000 / 754215
Processed 550000 / 754215
Processed 600000 / 754215
Processed 650000 / 754215
Processed 700000 / 754215
Processed 750000 / 754215


In [12]:
import pandas as pd

counts = np.array([len(v) for v in groups.values()])

stats = {
    "min": np.min(counts),
    "max": np.max(counts),
    "mean": np.mean(counts),
    "median": np.median(counts),
    "q1": np.quantile(counts, 0.25),
    "q2": np.quantile(counts, 0.50),  # median
    "q3": np.quantile(counts, 0.75),
    "std": np.std(counts),
}

stats_df = pd.DataFrame(stats.items(), columns=["Statistic", "Value"])

print(stats_df)

  Statistic      Value
0       min   1.000000
1       max  54.000000
2      mean  11.879085
3    median  14.000000
4        q1   4.000000
5        q2  14.000000
6        q3  18.000000
7       std   7.816462


In [ ]:
from run_real_data.models import HCDFlowResMLP

6


In [ ]:
model_path = ""
model = HCDFlowResMLP.load_from_checkpoint(model_path)
model.load_state_dict(torch.load(model_path))
model.eval()

In [ ]:
# sampling
import run_real_data.config as C
import run_real_data.preprocess as prep
intensity = model.sample(noise, pep, charge, C.ODE_STEPS)

In [8]:
print(len(groups.keys()))

63491
